# 01 — Prepare dataBuilds the tile archive from the three source scenes.Each scene is tiled **in its own coordinate system** — the images are neverwarped onto a common grid, only resampled to 0.125 m/px. The labels thereforesit exactly where they were drawn. The only thing moved between systems is thecorridor polygon.Output per tile:```<scene>_<col>_<row>.tif         image, uint16, raw values<scene>_<col>_<row>_masks.tif   label raster, one integer per boulderindex.json                      bounds, fold, counts per tile```Runs **locally** — the source scenes are on the laptop. Only the finishedarchive goes to the server.

In [ ]:
import sys, jsonimport numpy as npimport geopandas as gpdimport rasterioimport matplotlib.pyplot as pltsys.path.insert(0, ".")import configfrom common import tiles, foldsconfig.make_dirs()print("raw     ", config.RAW)print("archive ", config.ARCHIVE)print("res     ", config.RES, "m/px |  tile", config.TILE, "px =",      config.TILE * config.RES, "m |  overlap", config.OVERLAP, "px")

## Source scenes

In [ ]:
rows = []for s, info in config.SCENES.items():    p = config.image(s)    with rasterio.open(p) as r:        rows.append((s, info["date"], p.name, r.count, f"{r.res[0]:.4f}",                     f"{r.width} x {r.height}", str(r.dtypes[0]),                     f"{config.RES / r.res[0]:.2f}x"))print(f"{'scene':6}{'date':12}{'file':28}{'b':>2} {'m/px':>8} {'size':>16} {'type':>8} {'upsample':>9}")for r in rows:    print(f"{r[0]:6}{r[1]:12}{r[2]:28}{r[3]:>2} {r[4]:>8} {r[5]:>16} {r[6]:>8} {r[7]:>9}")

## Outlines and size distribution

In [ ]:
out = {}for s in config.SCENES:    g = gpd.read_file(config.OUTLINES / f"{s}_bloecke.gpkg").to_crs(config.CRS)    g["d_m"] = 2 * np.sqrt(g.area / np.pi)    out[s] = g    ok = g.d_m >= config.MIN_DIAMETER    tr = g.d_m >= config.MIN_TRAINING    print(f"{s}: {len(g):5d} total | {tr.sum():5d} >= {config.MIN_TRAINING} m (training)"          f" | {ok.sum():5d} >= {config.MIN_DIAMETER} m (evaluated)"          f" | D50 {g.d_m[ok].median():.2f} m | max {g.d_m.max():.1f} m")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))for s, g in out.items():    ax.hist(g.d_m[g.d_m < 15], bins=60, histtype="step", label=s, lw=1.4)ax.axvline(config.MIN_DIAMETER, color="k", ls="--", lw=1)ax.set_xlabel("equivalent diameter (m)"); ax.set_ylabel("count"); ax.legend()ax.set_title(f"size distribution, dashed = evaluation limit {config.MIN_DIAMETER} m")plt.tight_layout(); plt.show()

## Overlapping polygonsA label raster holds one integer per pixel, so overlapping outlines cannot berepresented — the later one would overwrite the earlier. Check how bad it is.

In [ ]:
for s, g in out.items():    g2 = g[g.d_m >= config.MIN_TRAINING].reset_index(drop=True)    pairs = gpd.sjoin(g2[["geometry"]], g2[["geometry"]], predicate="overlaps")    n = len(pairs) // 2    if n:        a = g2.geometry.values[pairs.index.values]        b = g2.geometry.values[pairs.index_right.values]        frac = np.array([x.intersection(y).area / x.area for x, y in zip(a, b)])        print(f"{s}: {n:4d} overlapping pairs | median overlap "              f"{np.median(frac) * 100:.1f} % of the smaller one, max {frac.max() * 100:.1f} %")    else:        print(f"{s}:    0 overlapping pairs")

## Fold assignment

In [ ]:
print(f"test block      km {config.TEST_FROM} - {config.TEST_TO}")print(f"fold edges      {config.FOLD_EDGES}")print(f"always training {config.ALWAYS_TRAIN}\n")order = ["always_train", "fold1", "fold2", "fold3", "fold4", "test"]print(f"{'scene':6}" + "".join(f"{k:>14}" for k in order))total = {k: 0 for k in order}for s in config.SCENES:    nov = gpd.read_file(config.outlines(s)).to_crs(config.CRS)    d = 2 * np.sqrt(nov.area / np.pi)    nov = nov[d >= config.MIN_DIAMETER]    c = folds.summary(nov)    for k in order:        total[k] += c.get(k, 0)    print(f"{s:6}" + "".join(f"{c.get(k, 0):>14}" for k in order))print(f"{'all':6}" + "".join(f"{total[k]:>14}" for k in order))

## Build the archiveTakes a few minutes per scene — it reads and resamples roughly 150 Mpx each.

In [ ]:
index = []for s in config.SCENES:    print(f"\n{s}:")    index += tiles.build_scene(s)p = tiles.write_index(index)print(f"\n{len(index)} tiles total -> {p}")

## Archive overview

In [ ]:
import collectionsidx = tiles.read_index()print(f"{'scene':6}{'tiles':>7}{'with labels':>13}{'boulders':>10}{'median/tile':>13}")for s in config.SCENES:    t = [x for x in idx if x["scene"] == s]    lab = [x for x in t if x["n_inner"] > 0]    n = [x["n_inner"] for x in lab]    print(f"{s:6}{len(t):>7}{len(lab):>13}{sum(n):>10}"          f"{(np.median(n) if n else 0):>13.0f}")print()c = collections.Counter(x["fold"] for x in idx)print("tiles per fold:", dict(sorted(c.items())))print("empty tiles   :", sum(1 for x in idx if x["n_inner"] == 0),      f"({sum(1 for x in idx if x['n_inner'] == 0) / len(idx) * 100:.0f} %)")

## Visual check

In [ ]:
idx = tiles.read_index()pick = [x for x in idx if x["n_inner"] >= 8][:6]fig, axes = plt.subplots(2, 3, figsize=(13, 9))for ax, t in zip(axes.ravel(), pick):    with rasterio.open(config.ARCHIVE / f"{t['name']}.tif") as r:        img = r.read(1).astype(float)    with rasterio.open(config.ARCHIVE / f"{t['name']}_masks.tif") as r:        lab = r.read(1)    g = img > 0    lo, hi = np.percentile(img[g], config.PERCENTILE) if g.any() else (0, 1)    ax.imshow(np.clip((img - lo) / max(hi - lo, 1), 0, 1), cmap="gray")    ax.contour(lab > 0, levels=[0.5], colors="red", linewidths=0.7)    ax.set_title(f"{t['name']}  {t['n_inner']} boulders  [{t['fold']}]", fontsize=9)    ax.set_xticks([]); ax.set_yticks([])plt.tight_layout(); plt.show()

## Send to the server```bashrsync -av --progress \  "/c/Users/pauls/Desktop/Potsdam_Master/Internship_Uni/trishuli_netz_data/archive/" \  pcpool:/DATA/schrepel/trishuli_daten/archive/```